# Get Crypto Addresses
## Written without Claude!
**Author:** Elisa Warner  
**Point of Code:** Let's try to get some reported ethereum addresses related to pig butchering scam

In [30]:
import requests
from pathlib import Path
import os
import json
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
import re

In [6]:
CACHE_FILE = "cache.json"

In [7]:
class Session(object):
    """
    Session class helps us to connect online

    Methods:
     - load_cache : load website cache for curl downloads
     - call_site_curl : download website html with requests package (good for simple sites)
     - get_call_url : combines query parameters with the base url to get exact website address
     - call_site_broweser : selenium-based driver loads website (for more sophisticated sites)
     - save_cache : save the cache for requests-based downloads
    """
    def __init__(self, cache_file = CACHE_FILE):
        self.cache_file = CACHE_FILE
        self.cache = self.load_cache()
        self.driver = webdriver.Chrome()
        
    def load_cache(self):
        if Path(self.cache_file).exists():
            with open(self.cache_file) as f:
                temp = f.read()
                cache = json.loads(temp)
                print("Load cache from memory")
        else:
            cache = {}
            print("Create new cache")
        return cache
        
    def call_site_curl(self, base_url, params={}):
        call_url = self.get_call_url(base_url, params)
        
        if call_url not in self.cache:
            response = json.loads(requests.get(call_url))
            self.cache[call_url] = response.text
            self.save_cache()
        else:
            print("Loading from memory...")
            response = self.cache[call_url]
        
        return response

    def get_call_url(self, base_url, params={}):
        call_url = base_url
        if params:
            call_url = base_url + "?"
            for p in params:
                call_url += str(p) + "=" + str(params[p])
                if p != list(params)[-1]:
                    call_url += "&"
        print("Calling:", call_url)
        return call_url

    def call_site_browser(self, base_url, params = {}):
        call_url = self.get_call_url(base_url, params)
        
        self.driver.get(call_url)
    
        # 3. Maximize window (Optional)
        self.driver.maximize_window()

        return self.driver
        
    def save_cache(self):
        if self.cache:
            with open(self.cache_file, "wb") as f:
                cache_str = json.dumps(self.cache)
                f.write(cache_str.encode())

In [8]:
def parse_page(page):
    soup = BeautifulSoup(page)
    soup.findall()
    return soup

#### We try to load page with curl, but it is unsuccessful

In [9]:
# load cache
CACHE_FILE = "cache.json"
session = Session()

Load cache from memory


In [54]:
base_url = "https://chainabuse.com/chain/ETH"
params = {"page":0, "filter":"PIGBUTCHERING"}

pattern = """Reported[A-Za-z0-9.=()_]+"""
text = session.call_site_curl(base_url, params)
re.findall(pattern, text)

Calling: https://chainabuse.com/chain/ETH?page=0&filter=PIGBUTCHERING
Loading from memory...


[]

#### Selenium Based Search Is Successful

In [61]:
base_url = "https://chainabuse.com/chain/ETH"

In [82]:
addresses = []
for page in range(1000):
    try:
        params = {"page":page, "filter":"PIGBUTCHERING"}
        
        driver = session.call_site_browser(base_url, params)
        time.sleep(2)
        search_box = driver.find_elements(By.XPATH, "//div[contains(@class, 'create-ReportedSection variant-default')]")
        
        for s in search_box:
            lines = s.text.split("\n")
            for l in lines:
                if l.startswith("0x"):
                    addresses.append(l)
    except:
        continue

Calling: https://chainabuse.com/chain/ETH?page=0&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=1&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=2&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=3&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=4&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=5&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=6&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=7&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=8&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=9&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=10&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=11&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=12&filter=PIGBUTCHERING
Calling: https://chainabuse.com/chain/ETH?page=13&filter=PIGBUTCHERING
Calling: https:/

In [83]:
len(addresses)

228

In [85]:
import pickle as pkl
with open("addresses.pkl","wb") as f:
    pkl.dump(addresses,f)

In [87]:
import pickle as pkl
with open("addresses.pkl","rb") as f:
    addresses = pkl.load(f)

In [88]:
addresses[1]

'0x7174d846b27fd468853303895aBb8dbE95E44808'